# 🔢 量化基础 — 推理视角的量化原理

**前置阅读**：建议先读完 Transformer 推理基础，理解模型权重和 KV Cache 在显存中的占比。

**本文目标**：建立量化的统一 mental model——不讲完整的量化数学理论，而是聚焦在**推理框架中为什么需要量化、不同的量化方案各自解决什么问题**。

读完这篇你会理解：
- 量化在做什么：从 FP16 到 INT4 的信息压缩
- 对称 vs 非对称量化、per-tensor / per-channel / per-group 的区别
- KV Cache 量化为什么是"免费午餐"
- GGUF (llama.cpp)、AWQ、GPTQ、FP8 — 各自的适用场景

## 1. 量化的本质

### 一张图理解量化

量化就是把浮点数"压缩"到更小的数据类型：

```
FP16 (2 字节, ~65K 个可能值):
  ...  -0.73  0.00  0.15  0.42  0.88  1.23  2.71  ...

         ↓ 量化 (scale=0.1, zero_point=0)

INT4 (0.5 字节, 16 个可能值):
  0  1  2  3  4  5  6  7  8  9  10 11 12 13 14 15
  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓
  ...  -0.7  ...  0.1  0.4  ...  0.9  1.2  ...  2.7  ...
       ↑              ↑         ↑              ↑
    最接近 -0.73   0.15→0.1   0.88→0.9    2.71→2.7
                (损失 0.05)            (损失 0.01)
```

**核心公式**：
```
量化:   q = round(x / scale) + zero_point
反量化: x' = (q - zero_point) * scale

其中:
  scale = (max(x) - min(x)) / (2^bits - 1)
  zero_point = round(-min(x) / scale)
```

### 为什么量化在推理中特别重要？

| 原因 | 说明 |
|------|------|
| **权重主导显存** | 14B 参数 × 2 字节(FP16) = 28 GB，一张 80GB A100 已经很紧张 |
| **KV Cache 线性增长** | 前面算过——10 并发 4K 请求，KV Cache 轻松超 20GB |
| **内存带宽瓶颈** | Decode 阶段 memory-bound，每次读权重都占带宽，压缩权重 = 加速 |
| **消费级硬件** | 在 MacBook/RTX 4060 (8GB) 上跑 7B 模型 → **必须量化** |

## 2. 量化的三个维度

理解量化方案的区别，看三个维度就够了：

### 2.1 对称量化 vs 非对称量化

```
对称量化（zero_point=0）:
  FP16: -0.5  0.0  0.3  0.8
         ↓  scale=0.1, zp=0
  INT4: 将 [-0.8, 0.7] 映射到 [0, 15]，0 点对齐

非对称量化（zero_point≠0）:
  FP16: 1.2  1.5  1.8  2.1  2.5 （全是正数）
         ↓  scale=0.1, zp=12
  INT4: 将 [1.2, 2.7] 映射到 [0, 15]，不浪费表示范围
```

| | 对称 | 非对称 |
|---|---|---|
| 适用场景 | 激活值（均值接近 0） | 某些权重（全正或全负分布） |
| 实现复杂度 | 低（少一个 zp 项） | 高 |
| 表示效率 | 数据分布不对称时浪费 | 充分利用位数 |
| llama.cpp 使用 | Q4_0, Q5_0 | Q4_K_M, Q5_K_M |

### 2.2 量化粒度

量化粒度决定了 **scale 和 zero_point 的共享范围**：

```
Per-Tensor（一个 tensor 用一组 scale/zp）:
  W [4096 × 4096] → 所有元素共享同一个 scale, zero_point
  优点: 最快，scale 开销最小
  缺点: outlier 会拖累整个 tensor 的精度

Per-Channel（每行或每列一组 scale/zp）:
  W [4096 × 4096] → 每行一个 scale/zp (共 4096 组)
  优点: 更好的精度，是大多数量化的默认选择
  缺点: 多了 4096 个 scale 的开销

Per-Group（每 N 个元素一组 scale/zp）:
  W [4096 × 4096] → 每 128 个元素一组 (共 4096*32 组)
  优点: 精度最高，GGUF K-quant 的核心技术
  缺点: scale 存储开销更大
```

**为什么 llama.cpp 的 K-quant 效果好？**
因为它对"重要"的权重（outlier 多的行/列）用更细的粒度（更多 bit），对不重要的一带而过——这是不对称的量化策略。

## 3. 主流推理量化方案对比

### 3.1 GGUF / GGML 量化 (llama.cpp)

**设计哲学**：在 CPU 上高效运行，精度尽量不损

```
格式      bits_per_weight  精度目标
Q4_0      4.5              legacy, 简单对称量化
Q4_K_M    4.8              质量和速度的最佳平衡 ★推荐
Q5_K_M    5.5              更好的质量
Q6_K      6.6              接近无损
Q8_0      8.5              几乎等于 FP16
IQ4_NL    4.3              极端压缩（importance-weighted）

K-quant 的核心技巧:
  1. 每 256 个元素一组 scale → 精度高
  2. 大权重（super-block）用更多 bit
  3. 小权重（sub-block）用更少 bit
  4. 专门的 outlier 处理
```

### 3.2 AWQ (Activation-Aware Weight Quantization)

**核心洞察**：不是所有权重同等重要——那些对应"显著激活通道"的权重需要更高精度。

```
传统方法: 所有通道同精度 → outlier 通道精度差 → 整体质量下降
AWQ:     找到"特别重要"的 ≈1% 通道 → 保留高精度 → 整体质量不降

实际做法:
  1. 跑一批校准数据 → 统计每个通道的激活值大小
  2. 激活值大的通道 → 权重保留更多 bit
  3. 激活值小的通道 → 权重可以粗量化
```

### 3.3 GPTQ

**核心方法**：逐列贪心量化 + 误差补偿。

```
传统 round-to-nearest: 逐元素量化，后面元素的误差不管
GPTQ: 量化第 i 列 → 计算对未量化列的误差 → 补偿到第 i+1 列

  Step 1: quantize(col_0) → 得到误差 δ₀
  Step 2: 把 δ₀ 的影响"推"到 col_1, col_2, ... col_n
  Step 3: 重复，直到所有列量化完成
  
  效果: 在相同 bit width 下精度显著优于简单量化
  代价: 需要校准数据 + 较长的量化时间
```

### 3.4 FP8 (NVIDIA Hopper 原生)

和 INT 量化的根本不同：

```
INT8: 均匀分布, [−127, 127] × scale
FP8:  浮点分布, 有 exponent!  → 动态范围更大
  
  以 E4M3 为例:
    4 bit 指数 → 动态范围 2⁻⁶ 到 2⁷
    3 bit 尾数 → 精度 1/8
    
  优势:
  - Outlier 值不需要特殊的 scale 处理（exponent 自动适应）
  - 硬件原生支持（H100 的 FP8 Tensor Core）
  - TensorRT-LLM 通过 FP8 在 H100 上获得 ~2x 加速
```

### 3.5 方案对比速查

| 方案 | 典型 bit | 硬件 | 精度损失 | 推理加速 | 代表框架 |
|------|---------|------|---------|---------|---------|
| GGUF Q4_K_M | 4.8 | CPU / Apple Silicon | 低 | 2-3x (带宽) | llama.cpp |
| GGUF Q5_K_M | 5.5 | CPU / Apple Silicon | 极低 | 1.5-2x | llama.cpp |
| AWQ INT4 | 4.0 | NVIDIA GPU | 低 | 2-3x | vLLM, TGI |
| GPTQ INT4 | 4.0 | NVIDIA GPU | 低 | 2-3x | vLLM, TGI |
| FP8 | 8.0 | H100/L40S+ | 极低 | ~2x | TensorRT-LLM |
| INT8 (PTDQ) | 8.0 | 大部分 GPU | 极低 | 1.3-1.5x | TensorRT-LLM |
| IQ4_NL | 4.3 | CPU | 中 | 3x+ | llama.cpp |

## 4. KV Cache 量化："免费的午餐"

除了量化模型权重，**KV Cache 也可以量化**——而且通常收益更大：

```
为什么叫"免费午餐"？

  模型权重量化:    损失精度 → 可能影响输出质量
  KV Cache 量化:   损失的是"缓存的历史信息"的精度
                    ↓
                    Attention scores 的精度↓ 一点点
                    但 Attention 本身就是 softmax 后的加权
                    轻微精度损失几乎不改变哪个 token 被关注
```

### llama.cpp 中的 KV Cache 量化

```
KV Cache dtype:  --cache-type-k f16    (默认 FP16)
                  --cache-type-v f16     

                  --cache-type-k q8_0   (Q8_0 量化, 推荐)
                  --cache-type-v q8_0   (节省 ~50% KV Cache 显存)

                  --cache-type-k q4_0   (更激进, 可能在长上下文中降质)
                  --cache-type-v q4_0
```

### 量化 KV Cache 的影响

```python
# 不量化: 1 token KV = 2 × 32层 × 32头 × 128维 × 2字节 = 512 KB
# Q8_0:    1 token KV = 512KB / 2 = 256 KB
# Q4_0:    1 token KV = 512KB / 4 = 128 KB

# 对于 32K 上下文的 7B 模型：
# FP16 KV: 512KB × 32768 = 16 GB  → 单卡几乎不可能
# Q8_0 KV: 256KB × 32768 = 8 GB   → 可以在 80GB 卡上跑
# Q4_0 KV: 128KB × 32768 = 4 GB   → 甚至能在 24GB 卡上跑
```

## 5. 实验：量化对推理的影响

让我们用一个简单的例子感受量化如何影响矩阵乘法——这是推理中最核心的操作。

In [1]:
# 量化对矩阵乘法精度的影响

import numpy as np

def quantize_symmetric(x, bits):
    """对称量化: FP32 → INT(bits)"""
    x_min, x_max = x.min(), x.max()
    abs_max = max(abs(x_min), abs(x_max))
    scale = abs_max / (2**(bits-1) - 1)
    
    x_q = np.clip(np.round(x / scale), -(2**(bits-1)-1), 2**(bits-1)-1).astype(np.int32)
    return x_q, scale

def dequantize(x_q, scale):
    return x_q.astype(np.float32) * scale

# 生成随机矩阵 (模拟权重)
np.random.seed(42)
A = np.random.randn(128, 128).astype(np.float32)
B = np.random.randn(128, 128).astype(np.float32)

# FP32 参考结果
C_ref = A @ B

print("=== 量化对矩阵乘法精度的影响 ===\n")

for bits, name in [(8, "INT8"), (6, "INT6"), (4, "INT4"), (3, "INT3")]:
    # 量化 A 和 B
    A_q, A_scale = quantize_symmetric(A, bits)
    B_q, B_scale = quantize_symmetric(B, bits)
    
    # 反量化
    A_dq = dequantize(A_q, A_scale)
    B_dq = dequantize(B_q, B_scale)
    
    # 量化后的矩阵乘法
    C_q = A_dq @ B_dq
    
    # 误差
    mse = np.mean((C_ref - C_q) ** 2)
    max_err = np.max(np.abs(C_ref - C_q))
    cosine_sim = np.dot(C_ref.flatten(), C_q.flatten()) / (
        np.linalg.norm(C_ref.flatten()) * np.linalg.norm(C_q.flatten())
    )
    
    # 存储开销 (相对于 FP16)
    fp16_bytes = 128 * 128 * 2      # FP16
    quant_bytes = 128 * 128 * bits / 8 + 1 * 4  # 加了 scale 开销
    saving = (1 - quant_bytes / fp16_bytes) * 100
    
    print(f"{name}: MSE={mse:.6f}, MaxErr={max_err:.3f}, CosSim={cosine_sim:.4f}, "
          f"存储节省={saving:.0f}%, 实际bit={bits}")

print()
print("观察:")
print("  INT8 → 几乎无损 (CosSim > 0.9999)")
print("  INT4 → 轻微损失 (CosSim ~0.99)  ← 大多数框架的 sweet spot")
print("  INT3 → 明显退化 (CosSim << 0.99) ← 需要特殊技术 (如 K-quant)")


ModuleNotFoundError: No module named 'numpy'

## 6. 选型指南

```
你的场景是？

GPU 服务器 (NVIDIA A100/H100)?
├─ 追求极致吞吐 → TensorRT-LLM + FP8 (H100) / INT8 (A100)
├─ 灵活、多模型 → vLLM + AWQ INT4
└─ HuggingFace 生态 → TGI + GPTQ INT4

MacBook / Apple Silicon?
├─ 最佳平衡 → Q4_K_M (GGUF)
├─ 追求质量 → Q5_K_M / Q6_K
└─ 极端压缩 → IQ4_NL (但可能降质)

消费级 NVIDIA (RTX 4060 / 4070)?
├─ llama.cpp + CUDA + Q4_K_M (GGUF)
└─ 或用 Ollama (内置以上配置)

CPU only?
└─ llama.cpp + Q4_K_M → 最好的一刀切选择
```

> **经验法则**：不确定用什么 → 先用 **Q4_K_M (GGUF)** 或 **AWQ INT4**，99% 情况下这是性价比最佳的选择。

## 7. 下一步

现在你已经理解了量化的核心概念：

- ✅ FP16 → INT4 的数学原理
- ✅ 对称 vs 非对称、per-tensor / per-channel / per-group 的区别
- ✅ KV Cache 量化为什么是"免费午餐"
- ✅ GGUF / AWQ / GPTQ / FP8 各自的定位和适用场景

带着这些知识进入各框架的 deep dive：
- **llama.cpp** → 理解 GGUF 格式和 K-quant 的实现细节
- **vLLM** → 理解 AWQ 如何与 PagedAttention 配合
- **TensorRT-LLM** → 理解 NVIDIA 原生的 FP8/INT8 量化管线
